# Cuadernillo de analisis: Credit Approval (UCI)

## Contexto general
Este dataset corresponde a solicitudes de tarjeta de credito y su resultado final de aprobacion.
Los nombres de atributos y valores fueron anonimizados en origen para proteger confidencialidad.

- Instancias: 690
- Variables predictoras: 15 (mezcla de continuas y categoricas)
- Variable objetivo: `A16` (`+` aprobada, `-` rechazada)
- Valores faltantes: presentes en varias columnas (representados como `?`)

Este cuadernillo prepara el dataset para ejercicios de **clasificacion**, **regresion** (como practica) y **redes neuronales**.

## Diccionario de columnas

| Columna | Tipo original | Valores / rango | Significado practico |
|---|---|---|---|
| A1 | Categorica | `a`, `b` | Atributo binario anonimizado del solicitante |
| A2 | Continua | numerica | Variable continua anonimizada (ej. medida demografica/financiera) |
| A3 | Continua | numerica | Variable continua anonimizada |
| A4 | Categorica | `u`, `y`, `l`, `t` | Categoria nominal anonimizada |
| A5 | Categorica | `g`, `p`, `gg` | Categoria nominal anonimizada |
| A6 | Categorica | `c`, `d`, `cc`, ..., `ff` | Categoria nominal (mas cardinalidad) |
| A7 | Categorica | `v`, `h`, `bb`, ..., `o` | Categoria nominal |
| A8 | Continua | numerica | Variable continua anonimizada |
| A9 | Categorica binaria | `t`, `f` | Indicador booleano anonimizado |
| A10 | Categorica binaria | `t`, `f` | Indicador booleano anonimizado |
| A11 | Continua | numerica | Variable continua anonimizada |
| A12 | Categorica binaria | `t`, `f` | Indicador booleano anonimizado |
| A13 | Categorica | `g`, `p`, `s` | Categoria nominal anonimizada |
| A14 | Continua | numerica | Variable continua anonimizada |
| A15 | Continua | numerica | Variable continua anonimizada (tambien util para practica de regresion) |
| A16 | Objetivo | `+`, `-` | Clase final: aprobacion (`+`) o rechazo (`-`) |

**Nota:** por politicas de privacidad del dataset, no existe semantica publica exacta de cada atributo; se trabaja como problema tabular anonimo.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_absolute_error
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.neural_network import MLPClassifier

pd.set_option('display.max_columns', None)

In [ ]:
# Carga del dataset
columns = [f'A{i}' for i in range(1, 17)]
df = pd.read_csv('crx.data', header=None, names=columns)

print('Dimensiones:', df.shape)
display(df.head())

In [ ]:
# EDA inicial: tipos y faltantes
display(df.info())

missing_raw = (df == '?').sum()
print('Valores ? por columna:')
display(missing_raw[missing_raw > 0].sort_values(ascending=False))

## Plan de transformacion a numerico (y por que)

Para usar modelos de machine learning (sobre todo redes neuronales y regresion lineal/logistica), necesitamos entradas numericas y sin nulos:

1. **`?` -> `NaN`**
   - Los `?` son faltantes codificados como texto. Se convierten a `NaN` para poder imputarlos de forma estadistica.
2. **Separar columnas numericas y categoricas**
   - Numericas: `A2`, `A3`, `A8`, `A11`, `A14`, `A15`.
   - Categoricas: `A1`, `A4`, `A5`, `A6`, `A7`, `A9`, `A10`, `A12`, `A13`.
3. **Imputacion**
   - Numericas: mediana (robusta a outliers).
   - Categoricas: moda (valor mas frecuente).
4. **Codificacion de categoricas**
   - One-Hot Encoding para transformar texto a columnas binarias sin introducir orden artificial.
5. **Escalado de numericas**
   - Estandarizacion (`StandardScaler`) para que modelos sensibles a escala (redes neuronales, regresion) converjan mejor.
6. **Objetivo `A16`**
   - Se codifica `+ -> 1`, `- -> 0` para clasificacion binaria.

In [ ]:
# Limpieza base
df_clean = df.replace('?', np.nan).copy()

numeric_features = ['A2', 'A3', 'A8', 'A11', 'A14', 'A15']
categorical_features = ['A1', 'A4', 'A5', 'A6', 'A7', 'A9', 'A10', 'A12', 'A13']
target = 'A16'

# Forzamos numerico en columnas continuas
for col in numeric_features:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

# Codificacion de la variable objetivo
df_clean[target] = df_clean[target].map({'+': 1, '-': 0})

print('Nulos tras limpieza:')
display(df_clean.isna().sum()[df_clean.isna().sum() > 0].sort_values(ascending=False))
display(df_clean.head())

In [ ]:
# Preprocesador general para clasificacion / redes neuronales
num_pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipe, numeric_features),
        ('cat', cat_pipe, categorical_features),
    ]
)

X = df_clean.drop(columns=[target])
y = df_clean[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train_t = preprocessor.fit_transform(X_train)
X_test_t = preprocessor.transform(X_test)

print('Shape train transformado:', X_train_t.shape)
print('Shape test transformado:', X_test_t.shape)

## Preparacion para ejercicios

### 1) Clasificacion
Problema natural del dataset: predecir aprobacion (`A16`).

### 2) Redes neuronales
Se usa el mismo `X` preprocesado numericamente con escalado + one-hot.

### 3) Regresion (como practica)
Este dataset es originalmente de clasificacion. Para practicar regresion, proponemos usar `A15` como variable continua objetivo
(y excluir `A15` de las entradas) para construir un ejercicio de prediccion numerica.

In [ ]:
# Ejemplo rapido: clasificacion con Regresion Logistica
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_t, y_train)
pred = clf.predict(X_test_t)
print('Accuracy (LogisticRegression):', round(accuracy_score(y_test, pred), 4))

In [ ]:
# Ejemplo rapido: red neuronal para clasificacion
nn = MLPClassifier(hidden_layer_sizes=(32, 16), activation='relu',
                   max_iter=600, random_state=42)
nn.fit(X_train_t, y_train)
pred_nn = nn.predict(X_test_t)
print('Accuracy (MLPClassifier):', round(accuracy_score(y_test, pred_nn), 4))

In [ ]:
# Ejemplo de REGRESION (practica): predecir A15
reg_target = 'A15'
reg_features_num = ['A2', 'A3', 'A8', 'A11', 'A14']
reg_features_cat = categorical_features.copy()

df_reg = df_clean.dropna(subset=[reg_target]).copy()
X_reg = df_reg[reg_features_num + reg_features_cat]
y_reg = df_reg[reg_target]

reg_preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), reg_features_num),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), reg_features_cat),
    ]
)

Xr_train, Xr_test, yr_train, yr_test = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)
Xr_train_t = reg_preprocessor.fit_transform(Xr_train)
Xr_test_t = reg_preprocessor.transform(Xr_test)

reg = LinearRegression()
reg.fit(Xr_train_t, yr_train)
pred_reg = reg.predict(Xr_test_t)
print('MAE (LinearRegression sobre A15):', round(mean_absolute_error(yr_test, pred_reg), 4))

## Resumen de cambios realizados al dataset

- Se reemplazo `?` por `NaN` para tratar faltantes correctamente.
- Se convirtieron columnas continuas a tipo numerico (`float`).
- Se codifico `A16` de texto a binario (`+`=1, `-`=0).
- Se imputaron faltantes (mediana/moda segun tipo de variable).
- Se aplico One-Hot Encoding en variables categoricas para obtener una matriz totalmente numerica.
- Se estandarizaron variables numericas para mejorar estabilidad y rendimiento en modelos sensibles a escala.

Con esto, el dataset queda listo para tus ejercicios de modelado.